In [1]:
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('../')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [4]:
Epilepsy_Control_Lab1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Control_toStack")

In [5]:
Epilepsy_Control_Lab1.count()

15782777

In [6]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_pivoted_Control_Lab2_RPN")

In [7]:
Epilepsy_Control_Lab.count()

412634

In [4]:
# Epilepsy_Control_Lab_clean = Epilepsy_Control_Lab.where(
#     (Epilepsy_Control_Lab.labcode != 'ABSOLUTE GRANU') & 
#     (Epilepsy_Control_Lab.labcode != 'ABSOLUTE LYMPH') & 
#     (Epilepsy_Control_Lab.labcode != 'ANA WITH REFLE') & 
#     (Epilepsy_Control_Lab.labcode != 'Glucose 3 Hour Specimen') & 
#     (Epilepsy_Control_Lab.labcode != 'HEPATITIS C AN') &
#     (Epilepsy_Control_Lab.labcode != 'IRON BINDING C') &
#     (Epilepsy_Control_Lab.labcode != 'TSH WITH REFLE') &
#     (Epilepsy_Control_Lab.labcode != 'SEX HORMONE BI') &
#     (Epilepsy_Control_Lab.labcode != 'TESTOSTERONE F')
    
# )

from pyspark.sql.functions import col, when

# Define mapping from old values to new values
labcode_mapping = {
    "ABSOLUTE GRANU": "ABSOLUTE_GRANU",
    "ABSOLUTE LYMPH": "ABSOLUTE_LYMPH",
    "ANA WITH REFLE": "ANA_WITH_REFLE",
    "Glucose 3 Hour Specimen": "Glucose_3_Hour_Specimen",
    "HEPATITIS C AN": "HEPATITIS_C_AN",
    "IRON BINDING C": "IRON_BINDING_C",
    "SEX HORMONE BI": "SEX_HORMONE_BI",
    "TESTOSTERONE F": "TESTOSTERONE_F",
    "TSH WITH REFLE": "TSH_WITH_REFLE"
}

# Create a new column with the renamed labcode values
Epilepsy_Control_Lab = Epilepsy_Control_Lab.withColumn("new_labcode",
    when(col("labcode") == "ABSOLUTE GRANU", labcode_mapping["ABSOLUTE GRANU"])
    .when(col("labcode") == "ABSOLUTE LYMPH", labcode_mapping["ABSOLUTE LYMPH"])
    .when(col("labcode") == "ANA WITH REFLE", labcode_mapping["ANA WITH REFLE"])
    .when(col("labcode") == "Glucose 3 Hour Specimen", labcode_mapping["Glucose 3 Hour Specimen"])
    .when(col("labcode") == "HEPATITIS C AN", labcode_mapping["HEPATITIS C AN"])
    .when(col("labcode") == "IRON BINDING C", labcode_mapping["IRON BINDING C"])
    .when(col("labcode") == "SEX HORMONE BI", labcode_mapping["SEX HORMONE BI"])
    .when(col("labcode") == "TESTOSTERONE F", labcode_mapping["TESTOSTERONE F"])
    .when(col("labcode") == "TSH WITH REFLE", labcode_mapping["TSH WITH REFLE"])
    .otherwise(col("labcode"))
)

# Drop the original labcode column and rename the new_labcode column to labcode
Epilepsy_Control_Lab = Epilepsy_Control_Lab.drop("labcode").withColumnRenamed("new_labcode", "labcode")

▸,:,


In [5]:
Epilepsy_Control_Lab.printSchema()

▸,:,


root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [5]:
from pyspark.sql.functions import col

filtered_df = Epilepsy_Control_Lab.filter(col("labcode") == "ABSOLUTE_GRANU")
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+--------------------------+-----------+--------------+
|personid                            |New_updated_Interpretation|servicedate|labcode       |
+------------------------------------+--------------------------+-----------+--------------+
|6306b2ad-85c4-4d55-934a-82e4a24d1537|Normal                    |2014-05-02 |ABSOLUTE_GRANU|
|8d5d2f6a-99ae-4c08-b15a-e3994a43ea3e|Normal                    |2014-03-29 |ABSOLUTE_GRANU|
+------------------------------------+--------------------------+-----------+--------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
Epilepsy_Control_Lab.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_cleaned1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
df = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_cleaned1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
df.head(10)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[Row(labcode='13303-3', personid='43a60eef-9129-4bd1-8efd-b48431160b65', New_updated_Interpretation='Normal', servicedate='2021-07-19'),
 Row(labcode='15283-5', personid='6920d1db-ee4b-436d-b40c-c1ed04fa3689', New_updated_Interpretation='Normal', servicedate='2021-01-11'),
 Row(labcode='15283-5', personid='782fdae7-4c83-4145-9d7c-7717d90ebb26', New_updated_Interpretation='Normal', servicedate='2020-11-04'),
 Row(labcode='15283-5', personid='8abd0bb8-223e-4ca2-b12b-ba05e3571291', New_updated_Interpretation=None, servicedate='2021-08-09'),
 Row(labcode='15283-5', personid='8e74130d-5b2b-4017-a41c-56a1769673a1', New_updated_Interpretation=None, servicedate='2018-07-10'),
 Row(labcode='15283-5', personid='d40b9328-e94e-469b-bc32-4c3f9953bc34', New_updated_Interpretation='Normal', servicedate='2019-02-11'),
 Row(labcode='15283-5', personid='35759245-d609-46bb-a9f0-35b5bcce2e44', New_updated_Interpretation='Normal', servicedate='2022-04-20'),
 Row(labcode='15283-5', personid='e396aca0-1f1c-4

<IPython.core.display.Javascript object>

In [47]:
from pyspark.sql.functions import col

filtered_df = df.filter(col("labcode") == "ABSOLUTE GRANU")
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------+--------+--------------------------+-----------+
|labcode|personid|New_updated_Interpretation|servicedate|
+-------+--------+--------------------------+-----------+
+-------+--------+--------------------------+-----------+



<IPython.core.display.Javascript object>

In [15]:
from pyspark.sql.functions import col

filtered_df = df.filter((col("personid") == "6920d1db-ee4b-436d-b40c-c1ed04fa3689") & (col("labcode") == "15283-5"))
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------+------------------------------------+--------------------------+-----------+
|labcode|personid                            |New_updated_Interpretation|servicedate|
+-------+------------------------------------+--------------------------+-----------+
|15283-5|6920d1db-ee4b-436d-b40c-c1ed04fa3689|Normal                    |2021-01-11 |
+-------+------------------------------------+--------------------------+-----------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print(df.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

15782777


<IPython.core.display.Javascript object>

In [9]:
## No need of this code as already the New_updated_Interpretation are normalized based on the latest labtest data and mode if there is a tie
from pyspark.sql import functions as F
df_1 = df.groupby('personid','labcode').agg(F.first('New_updated_Interpretation').alias('New_updated_Interpretation'))

▸,:,


In [10]:
df_1.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)



In [11]:
df_1.head(10)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[Row(personid='000058c9-4684-4a8a-9913-6a6ba01f8208', labcode='33914-3', New_updated_Interpretation=None),
 Row(personid='000058c9-4684-4a8a-9913-6a6ba01f8208', labcode='62487-4', New_updated_Interpretation=None),
 Row(personid='000086b2-3048-43ce-a370-25dfde6b7fa7', labcode='789-8', New_updated_Interpretation='Normal'),
 Row(personid='0000ab7e-b948-4bf8-ba24-2671c68ac33b', labcode='751-8', New_updated_Interpretation='Normal'),
 Row(personid='000151d8-40ee-46d4-860a-a757a06ff3b7', labcode='706-2', New_updated_Interpretation='Normal'),
 Row(personid='00019068-80ee-4e89-884e-72a73e77eae7', labcode='786-4', New_updated_Interpretation='Normal'),
 Row(personid='0001cbda-90e1-46f8-8532-bd9d616ea90f', labcode='1759-0', New_updated_Interpretation='Normal'),
 Row(personid='0001cbda-90e1-46f8-8532-bd9d616ea90f', labcode='5778-6', New_updated_Interpretation='Normal'),
 Row(personid='00028d93-51ee-4cc7-9d27-3f0470853057', labcode='66746-9', New_updated_Interpretation=None),
 Row(personid='0002e84b

<IPython.core.display.Javascript object>

In [17]:
print(df_1.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

15782777


<IPython.core.display.Javascript object>

In [10]:
# No need of aggregation as the labcodes are already normalized to have only 1 interpretation per unique personid, labcode combination
df_1.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_agg1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
Epilepsy_Control_lab_agg = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_agg1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
from pyspark.sql.functions import col

filtered_df = Epilepsy_Control_lab_agg.filter(col("labcode") == "ABSOLUTE GRANU")
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-------+--------------------------+
|personid|labcode|New_updated_Interpretation|
+--------+-------+--------------------------+
+--------+-------+--------------------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Used the original DF and not aggregated
from pyspark.sql import functions as F
Epilepsy_Control_lab_pivoted = Epilepsy_Control_lab_agg.groupby('personid').pivot('labcode').agg(F.first('New_updated_Interpretation'))
# Epilepsy_Control_lab_pivoted = df.groupby('personid').pivot('labcode').agg(F.first('New_updated_Interpretation'))

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
Epilepsy_Control_lab_pivoted.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- 1004-1: string (nullable = true)
 |-- 1005-8: string (nullable = true)
 |-- 1006-6: string (nullable = true)
 |-- 1007-4: string (nullable = true)
 |-- 10328-3: string (nullable = true)
 |-- 10329-1: string (nullable = true)
 |-- 10331-7: string (nullable = true)
 |-- 10332-5: string (nullable = true)
 |-- 10333-3: string (nullable = true)
 |-- 10334-1: string (nullable = true)
 |-- 10335-8: string (nullable = true)
 |-- 10338-2: string (nullable = true)
 |-- 1034-8: string (nullable = true)
 |-- 10341-6: string (nullable = true)
 |-- 10346-5: string (nullable = true)
 |-- 10350-7: string (nullable = true)
 |-- 10352-3: string (nullable = true)
 |-- 10354-9: string (nullable = true)
 |-- 10360-6: string (nullable = true)
 |-- 10362-2: string (nullable = true)
 |-- 10365-5: string (nullable = true)
 |-- 10368-9: string (nullable = true)
 |-- 10371-3: string (nullable = true)
 |-- 10373-9: string (nullable = true)
 |-- 10374-7: string (nu

In [5]:
# Epilepsy_Control_lab_pivoted.head(5)

▸,:,


In [4]:
spark.conf.set('spark.sql.pivotMaxValues', u'10000')
spark.conf.set("spark.driver.memory", "16g")

▸,:,


In [5]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = Epilepsy_Control_lab_pivoted.rdd.getNumPartitions()
Epilepsy_Control_lab_pivoted_repart = Epilepsy_Control_lab_pivoted.repartition(reparNum)

▸,:,


In [6]:
Epilepsy_Control_lab_pivoted_repart.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_pivot1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
Epilepsy_Control_Lab_Pivoted = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya_control_lab_pivot1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from pyspark.sql.functions import col

# Select the column with name 'ABSOLUTE_GRANU'
selected_column_df = Epilepsy_Control_Lab_Pivoted.select("ABSOLUTE_GRANU")

# Display the DataFrame
selected_column_df.show(truncate=False)


▸,:,


<IPython.core.display.Javascript object>

+--------------+
|ABSOLUTE_GRANU|
+--------------+
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
|null          |
+--------------+
only showing top 20 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# import math

# def split_df(df, num_split):
#     total_rows = len(df.columns)
    
    
#     splitted = []
    
#     num_columns_per_split = math.ceil((total_columns - 1) / num_split)  # Subtract 1 to exclude the first column
    
#     for i in range(num_split):
#         start = 1 + i * num_columns_per_split  # Start from the second column
#         end = min(1 + (i + 1) * num_columns_per_split, total_columns)  # Add 1 to adjust for the first column
#         split_columns = [first_column] + df.columns[start:end]
#         split_df = df[split_columns]
#         splitted.append(split_df)
        
#     return splitted

In [9]:
# from pyspark.sql import DataFrame

# def divide_dataframe(df: DataFrame, num_div: int) -> list:
#     """
#     Divides a PySpark DataFrame into a list of smaller DataFrames.

#     Parameters:
#     df (DataFrame): The input DataFrame to be divided.
#     num_div (int): The number of divisions.

#     Returns:
#     list: A list of DataFrames.
#     """
#     # Calculate the number of rows per division
#     total_rows = df.count()
#     rows_per_div = total_rows // num_div
#     remainder = total_rows % num_div

#     dataframes = []
#     start_idx = 0

#     for i in range(num_div):
#         end_idx = start_idx + rows_per_div + (1 if i < remainder else 0)
#         partition_df = df.limit(end_idx).subtract(df.limit(start_idx))
#         dataframes.append(partition_df)
#         start_idx = end_idx

#     return dataframes

▸,:,


In [ ]:
# divided_df = divide_dataframe(Epilepsy_Control_lab_agg,10)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# def pivot_and_join(dataframes: list, pivot_column: str, value_column: str, join_column: str) -> DataFrame:
#     """
#     Pivots each DataFrame in the list and then joins them together.

#     Parameters:
#     dataframes (list): List of DataFrames to be pivoted and joined.
#     pivot_column (str): Column name to pivot on.
#     value_column (str): Column name whose values will fill the pivoted columns.
#     join_column (str): Column name to join the DataFrames on.

#     Returns:
#     DataFrame: The resulting joined DataFrame.
#     """
#     pivoted_dfs = []
    
#     for df in dataframes:
#         pivoted_df = df.groupBy(join_column).pivot(pivot_column).agg(F.first(value_column))
#         pivoted_dfs.append(pivoted_df)

#     # Join all the pivoted DataFrames together on the join column
#     result_df = pivoted_dfs[0]
#     for pivoted_df in pivoted_dfs[1:]:
#         result_df = result_df.join(pivoted_df, on=join_column, how='outer')

#     return result_df

# # Example usage:
# # df = spark.read.csv('path_to_csv')
# # divided_dfs = divide_dataframe(df, 5)
# # result_df = pivot_and_join(divided_dfs, 'pivot_column', 'value_column', 'join_column')

▸,:,


In [3]:
# epi_control_lab_split = split_df(Epilepsy_Control_lab_pivoted,10)

In [ ]:
# result_df = pivot_and_join(divided_df, 'labcode', 'New_updated_Interpretation', 'personid')

In [ ]:
# result_df.write.mode('overwrite').parquet('file:/home/z_han/work/Oklahoma State/Zheng_Han/epilepsy/priya_control_lab_pivot')

In [2]:
epi_control_lab_pivoted = spark.read.parquet('file:/home/z_han/work/Oklahoma State/Zheng_Han/epilepsy/priya_control_lab_pivot')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
epi_control_lab_pivoted.show(5)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+------+------+------+------+-------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------

<IPython.core.display.Javascript object>

In [4]:
epi_control_lab_pivoted.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- 1004-1: string (nullable = true)
 |-- 1005-8: string (nullable = true)
 |-- 1006-6: string (nullable = true)
 |-- 1007-4: string (nullable = true)
 |-- 10328-3: string (nullable = true)
 |-- 10329-1: string (nullable = true)
 |-- 10331-7: string (nullable = true)
 |-- 10332-5: string (nullable = true)
 |-- 10333-3: string (nullable = true)
 |-- 10334-1: string (nullable = true)
 |-- 10335-8: string (nullable = true)
 |-- 10338-2: string (nullable = true)
 |-- 1034-8: string (nullable = true)
 |-- 10341-6: string (nullable = true)
 |-- 10346-5: string (nullable = true)
 |-- 10350-7: string (nullable = true)
 |-- 10352-3: string (nullable = true)
 |-- 10354-9: string (nullable = true)
 |-- 10360-6: string (nullable = true)
 |-- 10362-2: string (nullable = true)
 |-- 10365-5: string (nullable = true)
 |-- 10368-9: string (nullable = true)
 |-- 10371-3: string (nullable = true)
 |-- 10373-9: string (nullable = true)
 |-- 10374-7: string (nu